<a href="https://colab.research.google.com/github/tirthas970-cmyk/Galaxy-Classifier-Using-ML/blob/main/Galaxy_Classifier_Copy_for_Portfolio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NOTE:
# Quick Background: This is a galaxy classifer. Essentially, I'm using 24,000 images from the AstroNN dataset to train my model to classify galaxies based on their shapes using computer visions
# This not a completed project, and is in actively in development
# Here you will see my developmental process from my intial model to my current model
# To run, please press the run all button --> Note that this will run all models, but the main model (i.e., my current one) is in the last cell

In [ ]:
!pip install astroNN

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.6 MB/s eta 0:00:00


In [ ]:
from astroNN.datasets import load_galaxy10sdss
import numpy as np
import tensorflow as tf
from keras import layers, models
from sklearn.model_selection import train_test_split

images, labels = load_galaxy10sdss()

Galaxy10.h5:  99%|█████████▊| 208M/210M [00:16<00:00, 14.8MB/s]

Downloaded Galaxy10 successfully to /root/.astroNN/datasets/Galaxy10.h5


Galaxy10.h5: 210MB [00:17, 12.1MB/s]                           


In [ ]:
#Normalize images into [0, 1]
images = images.astype(np.float32) / 255.0

#Number 5 does skew with data due to low amount of images
mask = (labels !=5)

images = images[mask]
labels = labels[mask]


# Split into 70/10/20
train_idx, test_idx = train_test_split(range(labels.shape[0]), test_size=0.2, random_state=42, stratify=labels)
train_idx, val_idx = train_test_split(train_idx, test_size=0.125, random_state=42, stratify=labels[train_idx])

# Create the actual datasets using the indices
train_images, train_labels = images[train_idx], labels[train_idx]
val_images, val_labels = images[val_idx], labels[val_idx]
test_images, test_labels = images[test_idx], labels[test_idx]

num_classes = 10

#Turns numpy array into tensorflow dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(1000).batch(16)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(16)

#Optimization
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

print(f"Remaining classes: {np.unique(labels)}")
print(f"New dataset size: {len(images)}")

Remaining classes: [0 1 2 3 4 6 7 8 9]
New dataset size: 21768


In [ ]:
#Build CNN --> FIRST CNN

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"
  layers.RandomRotation(factor=0.5),            # Full 360-degree rotation (1.0 = 2π)
  layers.RandomZoom(0.2),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                   # Account for different telescope sensors


  layers.Conv2D(32, (3, 3), activation='relu'),

  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), activation='relu'),

  layers.MaxPooling2D((2,2)),

  layers.Flatten(),
  #To prevent overfitting
  layers.Dropout(0.5),

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

model.fit(train_ds, validation_data=val_ds, epochs=10)

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


Epoch 1/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 19s 8ms/step - accuracy: 0.4234 - loss: 1.4039 - val_accuracy: 0.4699 - val_loss: 1.2702
Epoch 2/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.4985 - loss: 1.2191 - val_accuracy: 0.5613 - val_loss: 1.0834
Epoch 3/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.5338 - loss: 1.1417 - val_accuracy: 0.5719 - val_loss: 1.0283
Epoch 4/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.5609 - loss: 1.0904 - val_accuracy: 0.5769 - val_loss: 1.0201
Epoch 5/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.5938 - loss: 1.0342 - val_accuracy: 0.6514 - val_loss: 0.8920
Epoch 6/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.6171 - loss: 0.9859 - val_accuracy: 0.6706 - val_loss: 0.8694
Epoch 7/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.6308 - loss: 0.9599 - val_accuracy: 0.6559 - val_loss: 0.9503
Epoch 8/10
953/953 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.6533 - loss: 0.9131 - val_accuracy

In [ ]:
###Refinement to Data Preperation

from sklearn.utils import class_weight

#Just changed batch sized from 16 to 32
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(1000).batch(64)
val_ds = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(64)

#Optimization
train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

#Adding Class Weights to prevent overfitting on majority of data
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
sqrt_weights = np.sqrt(weights)

class_weight_dict = dict(zip(np.unique(train_labels), sqrt_weights))

print(class_weight_dict)




{np.uint8(0): np.float64(0.8358950857042893), np.uint8(1): np.float64(0.5879811821248868), np.uint8(2): np.float64(0.6199483703009626), np.uint8(3): np.float64(2.634107930621662), np.uint8(4): np.float64(1.2555278145504662), np.uint8(6): np.float64(2.0271219255315835), np.uint8(7): np.float64(1.4685666134326942), np.uint8(8): np.float64(1.6341196416280461), np.uint8(9): np.float64(2.1596091881935555)}


In [ ]:
###Refinement to Model

#Batchnormalization

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"
  layers.RandomRotation(factor=0.5),            # variations in rotations
  layers.RandomZoom(0.2),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors
  layers.RandomTranslation(0.1, 0.1),           #Create more variations


  layers.Conv2D(64, (3, 3), use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  #Prevents Overfitting
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.2),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)

model.fit(train_ds, validation_data=val_ds, epochs=50, class_weight=class_weight_dict)

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


239/239 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - accuracy: 0.4284 - loss: 1.3736 - val_accuracy: 0.3211 - val_loss: 4.3029
Epoch 2/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.4679 - loss: 1.2296 - val_accuracy: 0.3137 - val_loss: 2.0292
Epoch 3/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 47ms/step - accuracy: 0.4984 - loss: 1.1658 - val_accuracy: 0.1929 - val_loss: 2.6887
Epoch 4/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.5331 - loss: 1.1017 - val_accuracy: 0.4318 - val_loss: 1.5325
Epoch 5/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.5620 - loss: 1.0401 - val_accuracy: 0.3519 - val_loss: 1.5718
Epoch 6/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.5848 - loss: 1.0009 - val_accuracy: 0.3032 - val_loss: 2.6256
Epoch 7/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.6023 - loss: 0.9628 - val_accuracy: 0.5131 - val_loss: 1.2655
Epoch 8/50
239/239 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.6157 - loss: 0.9313 - val_accurac

In [ ]:
### More Refinement to Model


model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors
 # layers.RandomTranslation(0.05, 0.05),           #Create more variations


  layers.Conv2D(64, (3, 3), padding='same', use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.MaxPooling2D((2, 2)),

  layers.Flatten(), #Changed it back to flatten so that spacial dimensions info isn't lost
  layers.Dense(256, activation='relu'), #to catch more patterns
  layers.Dropout(0.5),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

Epoch 1/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 29s 94ms/step - accuracy: 0.3615 - loss: 2.0592 - val_accuracy: 0.3215 - val_loss: 5.5520 - learning_rate: 0.0010
Epoch 2/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 22s 90ms/step - accuracy: 0.4010 - loss: 1.4450 - val_accuracy: 0.3266 - val_loss: 2.3233 - learning_rate: 0.0010
Epoch 3/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - accuracy: 0.4013 - loss: 1.4120 - val_accuracy: 0.3872 - val_loss: 1.6037 - learning_rate: 0.0010
Epoch 4/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - accuracy: 0.4221 - loss: 1.3605 - val_accuracy: 0.4933 - val_loss: 1.2140 - learning_rate: 0.0010
Epoch 5/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 22s 91ms/step - accuracy: 0.4483 - loss: 1.3164 - val_accuracy: 0.4405 - val_loss: 1.3574 - learning_rate: 0.0010
Epoch 6/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 40s 89ms/step - accuracy: 0.4762 - loss: 1.2665 - val_accuracy: 0.4704 - val_loss: 1.5457 - learning_rate: 0.0010
Epoch 7/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 22s 91ms/step - accuracy: 0.5008 - l

In [ ]:
#See socres and how model is doing

import numpy as np
from sklearn.metrics import classification_report

all_labels = []
all_preds = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    all_labels.extend(labels.numpy())
    # Take the index of the highest probability
    all_preds.extend(np.argmax(preds, axis=1))

print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.42      0.52      0.46       346
           1       0.84      0.98      0.90       700
           2       0.85      0.90      0.87       629
           3       0.00      0.00      0.00        35
           4       0.58      0.96      0.73       153
           6       0.00      0.00      0.00        59
           7       0.67      0.07      0.13       112
           8       0.00      0.00      0.00        91
           9       0.00      0.00      0.00        52

    accuracy                           0.73      2177
   macro avg       0.37      0.38      0.34      2177
weighted avg       0.66      0.73      0.67      2177



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [ ]:
###fine tunning
#Stratify data


model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                  # Account for different telescope sensors

  layers.Conv2D(64, (3, 3), padding='same', use_bias=False),    #use bias is set to false as batch normilization cancels it out
  layers.BatchNormalization(),
  layers.Activation('swish'), #swish for more optimization
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),


  layers.GlobalAveragePooling2D(), # Testing out GlobalAveragePooling Again
  layers.Dense(256, activation='relu'), #to catch more patterns
  layers.Dropout(0.5),  #changed form .5 to .2 to make learning easier

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])     #Wanted to test without class_weights

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


239/239 ━━━━━━━━━━━━━━━━━━━━ 28s 100ms/step - accuracy: 0.4651 - loss: 1.3245 - val_accuracy: 0.3215 - val_loss: 2.7893 - learning_rate: 0.0010
Epoch 2/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 100ms/step - accuracy: 0.5794 - loss: 1.0783 - val_accuracy: 0.3578 - val_loss: 1.5839 - learning_rate: 0.0010
Epoch 3/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.6656 - loss: 0.8842 - val_accuracy: 0.6422 - val_loss: 0.8951 - learning_rate: 0.0010
Epoch 4/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.7171 - loss: 0.7609 - val_accuracy: 0.6105 - val_loss: 1.0980 - learning_rate: 0.0010
Epoch 5/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.7424 - loss: 0.6983 - val_accuracy: 0.7588 - val_loss: 0.6681 - learning_rate: 0.0010
Epoch 6/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.7515 - loss: 0.6737 - val_accuracy: 0.6798 - val_loss: 0.8095 - learning_rate: 0.0010
Epoch 7/30
239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.7496 - loss: 0.67

In [ ]:
# To Check scores of previous model

import numpy as np
from sklearn.metrics import classification_report

all_labels = []
all_preds = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    all_labels.extend(labels.numpy())
    all_preds.extend(np.argmax(preds, axis=1))

print(classification_report(all_labels, all_preds))

In [ ]:
from tensorflow.keras import regularizers
#Even More finetuning

model = models.Sequential([
  layers.RandomFlip("horizontal_and_vertical", input_shape=images.shape[1: ]), # Space has no "up" or "down"; crates varriations
  layers.RandomRotation(factor=0.05),            # variations in rotations
  layers.RandomZoom(0.05),                       # Galaxies vary in distance/size
  layers.RandomContrast(0.1),                    # Account for different telescope sensors

  layers.Conv2D(64, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),  #using kernal regularizer to prevent the model from just memorizing info
  layers.BatchNormalization(),
  layers.Activation('swish'), #swish for more optimization
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  #Stacked so it analyzes it more
  layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),

  layers.Conv2D(264, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.Conv2D(264, (3, 3), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4)),
  layers.BatchNormalization(),
  layers.Activation('swish'),
  layers.MaxPooling2D((2, 2)),


  layers.GlobalAveragePooling2D(),
  layers.Dense(256, activation='swish'), #to catch more patterns
  layers.Dropout(0.5),

  layers.Dense(num_classes, activation='softmax')

])

model.compile(
    optimizer='adam',

    loss='sparse_categorical_crossentropy',

    metrics=['accuracy']

)


lr_schedule = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # Cut learning rate in half when progress stalls
    patience=3,      # Wait 3 epochs before cutting
    min_lr=0.00001
)


model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[lr_schedule])

# Save the model
model.save("GalaxyClassifierModel.keras")
print("Model saved as GalaxyClassifierModel")